# Notebook 1 — 時間序列特徵工程

這份 notebook 面向剛開始接觸 AIOps、網路監控資料與 Jupyter Notebook 的學員。內容會把原始監控計數器整理成資料表和圖，讓你能先看懂資料，再進入異常偵測。

本節把網路設備輸出的原始計數器整理成後續可用的特徵。

```text
原始監控欄位
  -> 合併 In / Out 計數器
  -> 轉成比較容易解讀的特徵
  -> 用 rolling statistics 建立短期 baseline
  -> 畫圖確認資料行為
```

開始前請先閱讀 `aiops-anomaly-detection/getting-started/README.md`。設定腳本不內建專案名稱或使用者目錄；文件中的 `my-aiops-project` 和 `$HOME/.venvs/my-aiops-project` 只是範例，請替換成你自己的名稱與路徑。

如果你已經用其他 notebook 工具打開檔案，請確認 kernel 是你在 setup script 裡設定的 project name。

## 你會用到的欄位

| 欄位 | 初學者解讀 |
|---|---|
| `INOCTETS` / `OUTOCTETS` | 進入 / 離開介面的 bytes 數；可粗略理解成流量大小。 |
| `INUCASTPKTS` / `OUTUCASTPKTS` | 單播封包數；多數正常主機通訊都屬於這類。 |
| `INERRORS` / `OUTERRORS` | 有錯誤的封包數；常和線路、介面、硬體問題有關。 |
| `INDISCARDS` / `OUTDISCARDS` | 被設備丟棄的封包數；常和壅塞或 queue 滿了有關。 |
| `INBROADCASTPKTS` / `OUTBROADCASTPKTS` | broadcast 封包數；異常升高時可能是 broadcast storm。 |
| `INMULTICASTPKTS` / `OUTMULTICASTPKTS` | multicast 封包數；異常升高時要看是否符合業務行為。 |
| `INUNKNOWNPROTOS` | 設備無法辨識協定的封包數；可作為異常線索。 |


## Step 0 — 載入套件與資料

這一格做三件事。第一，載入 Python 套件。第二，設定圖表大小與字體。第三，讀取本課程已準備好的模擬監控資料。

請先執行這一格。若出現 `ModuleNotFoundError`，代表目前 kernel 沒有安裝必要套件。請回到 `getting-started/README.md`，用你自己的 project name、virtual environment directory、cache directory 重新執行 setup script。

執行成功後，你應該會看到資料筆數、port 數、時間範圍，以及前三列資料。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 8
plt.rcParams['axes.titlesize'] = 9
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7
plt.rcParams['legend.fontsize'] = 7

# 資料路徑：從 notebooks/ 往上三層到 aiops-anomaly-zero-to-hero/
ROOT     = Path('../../..')          # aiops-anomaly-zero-to-hero/
DATA_CSV = ROOT / 'data/synthetic/synthetic_rrd_metrics.csv'
EVT_CSV  = ROOT / 'data/synthetic/synthetic_event_catalog.csv'

df     = pd.read_csv(DATA_CSV, parse_dates=['timestamp'])
events = pd.read_csv(EVT_CSV,  parse_dates=['start_time', 'end_time'])

df = df.sort_values(['port_id', 'timestamp']).reset_index(drop=True)

print('資料筆數:', len(df))
print('Port 數:', df['port_id'].nunique())
print('時間範圍:', df['timestamp'].min(), '→', df['timestamp'].max())
df.head(3)

## Step 1 — 先看事件表

`events` 是本課程的 ground truth，也就是「我們事先知道哪些時間真的發生了事件」。真實工作現場通常不會有這麼乾淨的答案；這裡提供它，是為了讓你可以練習比較「資料看起來異常」和「事件真的發生」之間的差異。

請注意三個欄位：`event_type`、`port_id`、`start_time` / `end_time`。後面的紅色背景區塊就是用這些時間畫出來的。


In [ ]:
events

## Step 2 — 把原始計數器轉成特徵

原始欄位通常太細，初學者很難直接判讀。例如 `INOCTETS` 和 `OUTOCTETS` 分開看時，只知道進出方向；合併成 `traffic_total` 後，才比較像「這個 port 現在總流量多大」。

這一步建立後續偵測會用到的特徵：

| 特徵 | 怎麼算 | 初學者判讀 |
|---|---|---|
| `traffic_total` | `INOCTETS + OUTOCTETS` | 總流量。突然升高可能是大量傳輸或攻擊。 |
| `ucast_total` | `INUCASTPKTS + OUTUCASTPKTS` | 單播封包總量。可和流量一起看封包大小。 |
| `avg_packet_size` | `traffic_total / ucast_total` | 平均封包大小。小封包偏多可能是掃描；大封包偏多可能是備份或下載。 |
| `error_rate` | `errors_total / ucast_total` | 錯誤比例。比單看錯誤數更公平，因為不同 port 流量不同。 |
| `discard_rate` | `discards_total / ucast_total` | 丟棄比例。升高時常代表壅塞或設備壓力。 |
| `broadcast_ratio` | `broadcast_total / ucast_total` | broadcast 相對比例。同步升高要留意 broadcast storm。 |
| `multicast_ratio` | `multicast_total / ucast_total` | multicast 相對比例。要搭配服務情境判讀。 |

`safe_div` 的目的只是避免除以 0。這不是模型技巧，而是資料處理的基本防呆。


In [ ]:
def safe_div(a, b):
    """除法，避免除以 0"""
    return np.where(b > 0, a / b, 0.0)

# 合計
df['traffic_total']   = df['INOCTETS']       + df['OUTOCTETS']
df['ucast_total']     = df['INUCASTPKTS']    + df['OUTUCASTPKTS']
df['errors_total']    = df['INERRORS']        + df['OUTERRORS']
df['discards_total']  = df['INDISCARDS']      + df['OUTDISCARDS']
df['broadcast_total'] = df['INBROADCASTPKTS'] + df['OUTBROADCASTPKTS']
df['multicast_total'] = df['INMULTICASTPKTS'] + df['OUTMULTICASTPKTS']

# 衍生比率
df['avg_packet_size'] = safe_div(df['traffic_total'],   df['ucast_total'])
df['error_rate']      = safe_div(df['errors_total'],    df['ucast_total'])
df['discard_rate']    = safe_div(df['discards_total'],  df['ucast_total'])
df['broadcast_ratio'] = safe_div(df['broadcast_total'], df['ucast_total'])
df['multicast_ratio'] = safe_div(df['multicast_total'], df['ucast_total'])

print('衍生特徵計算完成')
df[['timestamp', 'port_id', 'traffic_total', 'avg_packet_size', 'error_rate', 'discard_rate']].head()

## Step 3 — 先選一個 port 畫圖

直接看整張大表很難學。這裡先固定看 `port-id7427`，因為它包含 queue congestion 與 link issue 事件，圖上會比較容易看到變化。

紅色區塊代表事件發生時間。請先不要急著判斷模型好不好；這一節的重點是學會把欄位和圖形行為連起來。

你可以稍後把 `PORT` 改成其他值，例如 `port-id7428` 或 `port-id7430`，比較不同 port 的形狀。


In [ ]:
def mark_events(ax, port_id, events):
    """在圖上標記事件時間區段"""
    for _, ev in events.iterrows():
        if ev['port_id'] in [port_id, 'MULTI']:
            ax.axvspan(ev['start_time'], ev['end_time'],
                       alpha=0.15, color='red', label=ev['event_type'])

PORT = 'port-id7427'
p = df[df['port_id'] == PORT].copy()

In [ ]:
fig, ax = plt.subplots()
ax.plot(p['timestamp'], p['INOCTETS'],  label='INOCTETS',  linewidth=1)
ax.plot(p['timestamp'], p['OUTOCTETS'], label='OUTOCTETS', linewidth=1)
mark_events(ax, PORT, events)
ax.set_title(f'{PORT} — 流量 In / Out (OCTETS)')
ax.set_ylabel('bytes')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(p['timestamp'], p['avg_packet_size'], color='tab:purple', linewidth=1)
mark_events(ax, PORT, events)
ax.set_title(f'{PORT} — 平均封包大小 (avg_packet_size)')
ax.set_ylabel('bytes / packet')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(p['timestamp'], p['error_rate'],   label='error_rate',   linewidth=1)
ax.plot(p['timestamp'], p['discard_rate'], label='discard_rate', linewidth=1)
mark_events(ax, PORT, events)
ax.set_title(f'{PORT} — Error Rate & Discard Rate')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## Step 4 — 用 rolling 統計建立 baseline

`rolling` 的意思是「每一個時間點都回頭看最近一段時間」。例如 rolling mean 1h 代表：在目前這個時間點，用過去 1 小時的資料算平均。

這種 baseline 很適合初學者，因為它可解釋：如果現在的流量遠高於最近一小時的平均，就值得注意。

| 統計量 | 這裡的窗口 | 用途 |
|---|---:|---|
| `traffic_mean_1h` | 1 小時 | 最近正常水準的大概位置。 |
| `traffic_std_1h` | 1 小時 | 最近波動有多大。波動越大，正常範圍也越寬。 |
| `traffic_p95_6h` | 6 小時 | 過去 6 小時中偏高但仍常見的上界。 |

`bfill` 是用來補最前面 rolling window 還不夠長時產生的空值。這是教學上的簡化；正式系統要更嚴格地處理 warm-up 區間。


In [ ]:
# 以 port-id7427 示範，逐 port 計算
frames = []
for port_id, g in df.groupby('port_id'):
    g = g.set_index('timestamp').sort_index()
    g['traffic_mean_1h'] = g['traffic_total'].rolling('60min',  min_periods=3).mean()
    g['traffic_std_1h']  = g['traffic_total'].rolling('60min',  min_periods=3).std()
    g['traffic_p95_6h']  = g['traffic_total'].rolling('360min', min_periods=6).quantile(0.95)
    frames.append(g.reset_index())

df = pd.concat(frames, ignore_index=True)

# bfill 補齊最初幾筆 NaN（rolling warm-up 期間）
df[['traffic_mean_1h','traffic_std_1h','traffic_p95_6h']] = (
    df[['traffic_mean_1h','traffic_std_1h','traffic_p95_6h']].bfill()
)

print('Rolling 特徵計算完成')
df[['timestamp','port_id','traffic_total','traffic_mean_1h','traffic_p95_6h']].head()

## Step 5 — 把原始值和 baseline 疊在一起

這張圖是本 notebook 的核心檢查。你會同時看到：

- 原始 `traffic_total`：實際觀測到的流量。
- `rolling mean 1h`：最近一小時的平均水準。
- `rolling p95 6h`：過去六小時偏高的參考上界。
- 淺色區帶：`mean ± 3 std`，可粗略視為「最近波動下可接受的範圍」。

如果原始值長時間貼近或超過 baseline 上界，下一步才有理由做 anomaly detection。


In [ ]:
p = df[df['port_id'] == PORT].copy()

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(p['timestamp'], p['traffic_total'],    label='traffic_total (原始值)', linewidth=1, alpha=0.7)
ax.plot(p['timestamp'], p['traffic_mean_1h'],  label='rolling mean 1h',       linewidth=1.5)
ax.plot(p['timestamp'], p['traffic_p95_6h'],   label='rolling p95 6h (上限)',  linewidth=1.2, linestyle='--')

# mean ± 3 std 的 band
upper = p['traffic_mean_1h'] + 3 * p['traffic_std_1h']
lower = (p['traffic_mean_1h'] - 3 * p['traffic_std_1h']).clip(lower=0)
ax.fill_between(p['timestamp'], lower, upper, alpha=0.15, label='mean ± 3σ')

mark_events(ax, PORT, events)
ax.set_title(f'{PORT} — traffic_total 與 Rolling Baseline')
ax.set_ylabel('bytes')
ax.legend(loc='upper left', fontsize=7)
plt.tight_layout()
plt.show()

## 練習與檢查點

完成本 notebook 後，你應該能回答三個問題。

1. 把 `PORT` 改成 `port-id7428`、`port-id7430` 或其他 port，圖形有什麼不同？
2. queue congestion 發生時，`discard_rate` 是否明顯變化？如果沒有，可能代表什麼？
3. rolling window 太短會太敏感；太長會反應太慢。請用圖說明這句話，而不是只背定義。

下一份 notebook 會使用這裡的特徵與 baseline，建立第一個可解釋的 baseline anomaly detector。
